# FAD（CLAP music）— Colab GPU

ローカル CPU だと 1〜3 時間かかるので、ここ（GPU）で回す。

**事前準備（ローカル）**
1. `prttype/eval/fad/ref_wav` と `gen_wav` を zip する
2. Google Drive に置く（例: `MyDrive/fad_eval/fad_wav.zip`）

リポ取得は **学習ノート `colab_train.ipynb` と同じ**（`!git clone` → `repo` → `git pull`）。

ランタイムは **GPU**。Python **3.13.15** だと `fadtk` のメタデータ（`<=3.13`＝実質 3.13.0 まで）ではじかれる。ノートは `--ignore-requires-python` で回避するが、安定するなら fallback runtime（3.11）推奨。

In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # ※ Private リポジトリならここに PAT（repo 読取）を入れる

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

def authenticated_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    if repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@")
    return repo_url

CLONE_URL = authenticated_repo_url(REPO_URL, GITHUB_TOKEN)
print("clone:", REPO_URL)
print("token:", "set" if GITHUB_TOKEN else "not set")

# Drive 上の wav zip（中に ref_wav/ と gen_wav/）
WAV_ZIP = "/content/drive/MyDrive/fad_eval/fad_wav.zip"
REF_DIR = ""  # 展開済みならパスを入れる（その場合 zip は使わない）
GEN_DIR = ""
JSON_OUT = "/content/drive/MyDrive/fad_eval/fad_result.json"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("repo")

# 作業ディレクトリを /content に戻す（再実行時の安全策）
if not Path("scripts/compute_fad.py").exists() and Path("/content").exists():
    os.chdir("/content")

if not Path("scripts/compute_fad.py").exists():
    if REPO_DIR.exists() and not (REPO_DIR / "scripts" / "compute_fad.py").exists():
        print("不完全な repo を削除します")
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists() and (REPO_DIR / "scripts" / "compute_fad.py").exists():
        print("既存の repo を使用します")
        os.chdir(REPO_DIR)
    else:
        if not GITHUB_TOKEN:
            print("警告: Private リポジトリの場合、セル1に GITHUB_TOKEN を設定してください")
        !git clone {CLONE_URL} repo
        os.chdir("repo")

    if not Path("scripts/compute_fad.py").exists():
        raise FileNotFoundError(
            "clone 失敗。GITHUB_TOKEN を設定するか、ランタイムを再起動して再実行してください。"
        )

print("git pull ...")
!git pull
print("OK:", os.getcwd())
!git log --oneline -3

In [ ]:
import sys
print("python", sys.version)

# fadtk の Requires-Python "<=3.13" は 3.13.0 まで扱い。Colab の 3.13.15 は弾かれる
if sys.version_info >= (3, 13) and sys.version_info.micro > 0:
    print("NOTE: fadtk metadata excludes 3.13.x>0; installing with --ignore-requires-python")
    print("Prefer: Runtime → Use fallback runtime version (Python 3.11)")
    !pip install -q -U "numpy<2" "scipy<2" soundfile
    !pip install -q -U fadtk --ignore-requires-python
else:
    !pip install -q -U "numpy<2" "scipy<2" soundfile fadtk

import fadtk
import torch
print("fadtk ok")
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import zipfile
import shutil

WORK = Path("/content/fad_wav")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

ref = Path(REF_DIR) if REF_DIR else None
gen = Path(GEN_DIR) if GEN_DIR else None

if ref and gen and ref.is_dir() and gen.is_dir():
    print("using Drive dirs", ref, gen)
else:
    z = Path(WAV_ZIP)
    assert z.is_file(), f"zip がありません: {z}"
    with zipfile.ZipFile(z) as zf:
        zf.extractall(WORK)
    cands_ref = [p for p in WORK.rglob("ref_wav") if p.is_dir()]
    cands_gen = [p for p in WORK.rglob("gen_wav") if p.is_dir()]
    assert cands_ref and cands_gen, f"zip 内に ref_wav/gen_wav がありません: {list(WORK.iterdir())}"
    ref, gen = cands_ref[0], cands_gen[0]

n_ref = len(list(ref.glob("*.wav")))
n_gen = len(list(gen.glob("*.wav")))
print(f"ref={ref} ({n_ref})  gen={gen} ({n_gen})")
assert n_ref >= 1 and n_gen >= 1

In [ ]:
from pathlib import Path

out = Path(JSON_OUT)
out.parent.mkdir(parents=True, exist_ok=True)
!python scripts/compute_fad.py {ref} {gen} --json-out {out}
print(out.read_text(encoding="utf-8"))